<a href="https://colab.research.google.com/github/selinny/bist-algo-trading-analysis/blob/main/bist_analiz_ve_stratejiler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf

def hisse_rsi_durumu(hisse_kodu):
    veri = yf.download(hisse_kodu, period="6mo", progress=False)
    if veri.empty:
        return None

    fiyat = veri["Close"]
    fark = fiyat.diff()
    kazanc = fark.clip(lower=0)
    kayip = -1 * fark.clip(upper=0)

    ortalama_kazanc = kazanc.rolling(window=14).mean()
    ortalama_kayip = kayip.rolling(window=14).mean()

    rs = ortalama_kazanc / ortalama_kayip
    rsi = 100 - (100 / (1 + rs))

    guncel_fiyat = float(fiyat.iloc[-1])
    guncel_rsi = float(rsi.iloc[-1])


    if guncel_rsi < 30:
        karar = "🟢 AL (Aşırı Satım / Ucuz)"
    elif guncel_rsi > 70:
        karar = "🔴 SAT (Aşırı Alım / Pahalı)"
    else:
        karar = "⚪ NÖTR (Bekle)"

    return {
        "Hisse": hisse_kodu,
        "Son Fiyat": round(guncel_fiyat, 2),
        "RSI": round(guncel_rsi, 2),
        "Sinyal": karar
    }


takip_listesi = ["YKBNK.IS", "THYAO.IS", "ASELS.IS", "SISE.IS", "GARAN.IS"]

rapor = []
for hisse in takip_listesi:
    durum = hisse_rsi_durumu(hisse)
    if durum:
        rapor.append(durum)


pd.DataFrame(rapor)

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf

def bollinger_tarayici(hisse_kodu):
    veri = yf.download(hisse_kodu, period="6mo", progress=False)
    if veri.empty:
        return None

    fiyat = veri["Close"]


    orta_bant = fiyat.rolling(window=20).mean()
    std = fiyat.rolling(window=20).std()

    ust_bant = orta_bant + (2 * std)
    alt_bant = orta_bant - (2 * std)

    son_fiyat = float(fiyat.iloc[-1])
    son_ust = float(ust_bant.iloc[-1])
    son_alt = float(alt_bant.iloc[-1])


    if son_fiyat <= son_alt:
        sinyal = "🟢 AL (Alt Bandı Kırdı / Aşırı Ucuz)"
    elif son_fiyat >= son_ust:
        sinyal = "🔴 SAT (Üst Bandı Kırdı / Aşırı Pahalı)"
    else:
        sinyal = "⚪ NÖTR (Kanal İçinde)"

    return {
        "Hisse": hisse_kodu,
        "Son Fiyat": round(son_fiyat, 2),
        "Alt Bant": round(son_alt, 2),
        "Üst Bant": round(son_ust, 2),
        "Bollinger Sinyali": sinyal
    }

takip = ["YKBNK.IS", "THYAO.IS", "ASELS.IS", "SISE.IS"]
sonuclar = [bollinger_tarayici(h) for h in takip if bollinger_tarayici(h)]
pd.DataFrame(sonuclar)